# Scale selection for AR(p) zonotopes with uniform innovations

This notebook focuses only on the finite-history support zonotopes associated with a uniform-innovation AR(p) model.

For each embedding scale \(k\), the sliding-window vector

\[
Y_t^{(k)}=(X_t,\ldots,X_{t+k-1})\in\mathbb R^k
\]

has a finite-history zonotope approximation \(Z_k\), generated by the AR impulse response and the uniform innovation half-width.

The notebook embeds every \(Z_k\) into the common ambient space \(\mathbb R^{k_{\max}}\) by zero-padding,

\[
\phi_k(x_1,\ldots,x_k)=(x_1,\ldots,x_k,0,\ldots,0),
\]

then fills a scale similarity matrix from intrinsic-volume interactions,

\[
\Sigma_{kl}=f(Z_k\oplus Z_l)-f(Z_k)-f(Z_l),
\]

or the sign-reversed paper convention. The principal eigenvector of \(\Sigma\) is used as the scale profile.


In [ ]:

import math
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")


## User-facing parameters

The default settings are intentionally modest, so the zonotope calculations run quickly. Increase `history` or `k_max` once the basic behavior looks right.

Notes:

- `intrinsic_order = 1` is usually uninformative for the default Minkowski-sum operation, because \(V_1\) is additive under generator concatenation.
- `similarity_sign="interaction_positive"` uses
  \[
  f(Z_k\oplus Z_l)-f(Z_k)-f(Z_l),
  \]
  which is the sign-flipped version of your written formula. This tends to produce a nonnegative affinity matrix for \(V_j, j\ge2\), making the principal eigenvector easier to interpret.
- Set `similarity_sign="paper_formula"` to use exactly the sign written in the draft.


In [ ]:
# AR process parameters
p = 3
innovation_half_width = 1.0
random_seed = 2026

# If custom_phi is None, stationary full AR(p) coefficients are generated from PACF values.
# Example custom_phi for AR(3): np.array([0.6, -0.25, 0.1])
custom_phi = None

# Zonotope / scale-selection parameters
k_max = 10
history = 30
intrinsic_order = 2
combination_operation = "minkowski_sum"       # "minkowski_sum" or "averaged_minkowski_sum"
functional_name = "intrinsic_volume"         # "intrinsic_volume", "log_intrinsic_volume", "box_normalized_intrinsic_volume"
similarity_sign = "interaction_positive"     # "interaction_positive" or "paper_formula"
zero_diagonal = True
exact_subset_limit = 250_000
monte_carlo_samples = 40_000


## Uniform AR coefficient helpers

We use partial autocorrelations to generate stable AR(p) coefficients when no custom coefficients are supplied. The stochastic assumption enters through the uniform innovation half-width used in the zonotope generators.


In [ ]:
def pacf_to_ar_coefficients(partial_autocorrelations):
    partial_autocorrelations = np.asarray(partial_autocorrelations, dtype=float)
    if np.any(np.abs(partial_autocorrelations) >= 1):
        raise ValueError("All partial autocorrelations must lie strictly between -1 and 1.")

    coefficients = np.array([], dtype=float)
    for kappa in partial_autocorrelations:
        previous = coefficients.copy()
        coefficients = np.empty(len(previous) + 1)
        if len(previous):
            coefficients[:-1] = previous - kappa * previous[::-1]
        coefficients[-1] = kappa
    return coefficients


def default_full_ar_coefficients(order, scale=0.55, decay=0.72):
    pacf = np.array([
        scale * (decay ** j) * ((-1) ** j)
        for j in range(order)
    ])
    return pacf_to_ar_coefficients(pacf)


def companion_matrix(phi):
    phi = np.asarray(phi, dtype=float)
    order = len(phi)
    matrix = np.zeros((order, order))
    matrix[0] = phi
    if order > 1:
        matrix[1:, :-1] = np.eye(order - 1)
    return matrix


def spectral_radius(phi):
    return float(np.max(np.abs(np.linalg.eigvals(companion_matrix(phi)))))


def check_stationarity(phi):
    radius = spectral_radius(phi)
    if radius >= 1:
        raise ValueError(f"The AR coefficients are not stationary. Spectral radius = {radius:.4f}")
    return radius


In [ ]:
phi = np.asarray(custom_phi, dtype=float) if custom_phi is not None else default_full_ar_coefficients(p)
if len(phi) != p:
    raise ValueError(f"Expected {p} AR coefficients, got {len(phi)}.")

radius = check_stationarity(phi)

print(f"AR({p}) coefficients:", np.round(phi, 5))
print(f"Spectral radius: {radius:.4f}")
print(f"Uniform innovation support: [-{innovation_half_width:g}, {innovation_half_width:g}]")



## Zonotope construction

With uniform innovations \(\varepsilon_t\in[-a,a]\), the finite-history embedded support has generators given by the AR impulse response. The generator convention here is

\[
Z=\sum_i[-1,1]g_i,
\]

so the innovation half-width \(a\) is included directly in the generator columns.


In [ ]:

def ar_impulse_response(phi, maximum_lag):
    phi = np.asarray(phi, dtype=float)
    order = len(phi)
    psi = np.zeros(maximum_lag + 1)
    psi[0] = 1.0
    for lag in range(1, maximum_lag + 1):
        psi[lag] = sum(
            phi[j - 1] * psi[lag - j]
            for j in range(1, min(order, lag) + 1)
        )
    return psi


def ar_window_zonotope_generators(phi, embedding_dimension, history, half_width=1.0):
    k = embedding_dimension
    psi = ar_impulse_response(phi, history + k + 5)
    columns = []

    for innovation_time in range(-history, k):
        column = np.zeros(k)
        for coordinate in range(k):
            lag = coordinate - innovation_time
            if lag >= 0:
                column[coordinate] = half_width * psi[lag]
        if np.linalg.norm(column) > 1e-13:
            columns.append(column)

    return np.column_stack(columns)


def embed_generators(generators, ambient_dimension):
    generators = np.asarray(generators, dtype=float)
    embedded = np.zeros((ambient_dimension, generators.shape[1]))
    embedded[: generators.shape[0], :] = generators
    return embedded


def combine_zonotopes(embedded_generators_a, embedded_generators_b, operation="minkowski_sum"):
    if operation == "minkowski_sum":
        return np.column_stack([embedded_generators_a, embedded_generators_b])
    if operation == "averaged_minkowski_sum":
        return 0.5 * np.column_stack([embedded_generators_a, embedded_generators_b])
    raise ValueError(f"Unknown combination operation: {operation}")


def build_embedded_zonotopes(phi, k_max, history, half_width=1.0):
    zonotopes = {}
    for k in range(1, k_max + 1):
        generators = ar_window_zonotope_generators(phi, k, history, half_width)
        zonotopes[k] = embed_generators(generators, k_max)
    return zonotopes



## Functionals \(f\)

The main implemented functional is the intrinsic volume \(V_j\). For a zonotope generated by the columns of \(G\),

\[
V_j(Z)=\sum_{|I|=j}\sqrt{\det(G_I^\top G_I)}.
\]

For larger determinant sums, the notebook automatically switches to Monte Carlo subset sampling.


In [ ]:

def parallelotope_j_volume(generator_subset):
    gram = generator_subset.T @ generator_subset
    sign, log_determinant = np.linalg.slogdet(gram)
    if sign <= 0:
        return 0.0
    return float(np.exp(0.5 * log_determinant))


def intrinsic_volume_of_zonotope(generators, order, exact_limit=250_000, monte_carlo_samples=40_000, seed=2026):
    generators = np.asarray(generators, dtype=float)
    ambient_dimension, number_of_generators = generators.shape

    if order == 0:
        return {"value": 1.0, "standard_error": 0.0, "exact": True}
    if order < 0 or order > ambient_dimension:
        return {"value": 0.0, "standard_error": 0.0, "exact": True}
    if order == 1:
        return {"value": float(np.linalg.norm(generators, axis=0).sum()), "standard_error": 0.0, "exact": True}

    total_subsets = math.comb(number_of_generators, order)
    if total_subsets <= exact_limit:
        value = float(sum(
            parallelotope_j_volume(generators[:, index_set])
            for index_set in combinations(range(number_of_generators), order)
        ))
        return {"value": value, "standard_error": 0.0, "exact": True}

    rng = np.random.default_rng(seed + 1009 * order + number_of_generators)
    sampled_values = np.empty(monte_carlo_samples)
    for sample in range(monte_carlo_samples):
        index_set = rng.choice(number_of_generators, size=order, replace=False)
        sampled_values[sample] = parallelotope_j_volume(generators[:, index_set])

    value = float(total_subsets * sampled_values.mean())
    standard_error = float(total_subsets * sampled_values.std(ddof=1) / np.sqrt(monte_carlo_samples))
    return {"value": value, "standard_error": standard_error, "exact": False}


def box_intrinsic_volume(generators, order):
    if order == 0:
        return 1.0
    side_lengths = np.sum(np.abs(generators), axis=1)
    if order > len(side_lengths):
        return 0.0
    return float(sum(
        np.prod(side_lengths[list(index_set)])
        for index_set in combinations(range(len(side_lengths)), order)
    ))


def zonotope_functional(generators, functional="intrinsic_volume", intrinsic_order=2,
                        exact_limit=250_000, monte_carlo_samples=40_000, seed=2026):
    result = intrinsic_volume_of_zonotope(
        generators,
        intrinsic_order,
        exact_limit=exact_limit,
        monte_carlo_samples=monte_carlo_samples,
        seed=seed,
    )
    value = result["value"]

    if functional == "intrinsic_volume":
        return value, result
    if functional == "log_intrinsic_volume":
        return float(np.log1p(value)), result
    if functional == "box_normalized_intrinsic_volume":
        denominator = box_intrinsic_volume(generators, intrinsic_order)
        return (value / denominator if denominator > 0 else 0.0), result

    raise ValueError(f"Unknown functional: {functional}")



## Scale similarity matrix and principal eigenvector

Two sign conventions are available:

- `paper_formula`:
  \[
  f(Z_k)+f(Z_l)-f(Z_k\oplus Z_l).
  \]
- `interaction_positive`:
  \[
  f(Z_k\oplus Z_l)-f(Z_k)-f(Z_l).
  \]

The second is just the sign-flipped interaction term. For intrinsic volumes with \(j\ge2\), it is often easier to interpret as a positive affinity.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), dpi=130)

image = axes[0].imshow(Sigma, cmap="viridis", aspect="equal")
axes[0].set_title(r"Scale similarity matrix $\Sigma^{(n)}$")
axes[0].set_xlabel("l")
axes[0].set_ylabel("k")
axes[0].set_xticks(range(len(scales)))
axes[0].set_yticks(range(len(scales)))
axes[0].set_xticklabels(scales)
axes[0].set_yticklabels(scales)
fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].plot(scales, principal_vector, marker="o", linewidth=2)
axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.6)
axes[1].set_title("Principal eigenvector over scales")
axes[1].set_xlabel("scale k")
axes[1].set_ylabel("eigenvector component")
axes[1].set_xticks(scales)

plt.tight_layout()
plt.show()



## What to vary next

Good first variations:

1. Change `intrinsic_order` from 2 to 3.  
   This makes the interaction more genuinely volumetric, but can be slower.

2. Compare `similarity_sign="paper_formula"` and `"interaction_positive"`.  
   The paper formula is exactly your draft expression; the sign-flipped version is often nicer for eigenvector-based affinity.

3. Compare `combination_operation="minkowski_sum"` and `"averaged_minkowski_sum"`.  
   Averaging removes some pure scale growth from the Minkowski sum.

4. Change the AR coefficients via `custom_phi`.  
   The principal eigenvector is usually more sensitive to dependence strength and coefficient pattern than to the formal order \(p\) alone.

One caveat: with the default Minkowski sum, \(V_1\) is additive, so the interaction matrix is essentially zero for `intrinsic_order=1`. Use \(V_2\) or higher for meaningful scale interactions.
